# Snæfellsjökull: true-colour satellite scenes

This notebook downloads **one real satellite scene per year**, in true colour and at the sensor's full resolution, for the years listed in `YEARS`.

It differs from `v0_satellite_annual_composites.ipynb` in three ways:

| | `v0_satellite_annual_composites.ipynb` | this notebook |
|---|---|---|
| Image | median of every scene in the summer | **one scene, from one day** |
| Clouds | masked out, which leaves holes | **not masked**. The clearest day is picked instead, so there are no holes |
| Resolution | 600 px thumbnail (~20 m per pixel) | **sensor's native pixels** (10 m for Sentinel-2) |

For each year it ranks every day with imagery in the season by how cloudy the glacier was, shows you the best candidates, and downloads the clearest one (or the one you pick).

## Which satellite is used for which year

| Years | Satellite | Pixel | Colour |
|---|---|---|---|
| 2017 → | Sentinel-2 (surface reflectance) | **10 m** | true colour |
| 2015–2016 | Sentinel-2 (top of atmosphere) | 10 m | true colour |
| 1984–2014 | Landsat 5 / 7 / 8 (surface reflectance) | 30 m | true colour |
| 1972–1983 | Landsat 1–3 MSS (top of atmosphere) | 60 m | **approximate**: MSS has no blue band |
| before Oct 1972 | none | | |

**There is no satellite imagery of Snæfellsjökull from 1970.** The first Earth-observation satellite, Landsat 1, launched in July 1972. Its first image of this area is from 3 October 1972. The first late-summer scenes (September 1973) show the glacier under cloud, and the nearest clear one is from **29 August 1980** (Landsat 2). With `USE_NEAREST_YEAR = True`, a request for 1970 returns that scene and says so.

For early years, **aerial photographs** are much better than satellites: `aerial_photos.ipynb` downloads Iceland's archive mosaics of 1959, 1979, 1985 and 1989 at 0.5–1 m, in black and white.

Free imagery tops out at Sentinel-2's 10 m. Sharper images (0.3–3 m) are commercial (Maxar, Planet, Airbus).

## Outputs

Written to `outputs/true_color/` next to this notebook. `<date>` is the day the scene was taken.

| Path | Contents |
|---|---|
| `snaefellsjokull_<year>_<sensor>_<date>.png` | True-colour image at native resolution, one image pixel per satellite pixel |
| `snaefellsjokull_<year>_<sensor>_<date>.tif` | Same scene as a GeoTIFF, UTM zone 27N (EPSG:32627). Bands `red, green, blue, nir`; reflectance × 10,000 as 16-bit integers; no-data `-32768`. For MSS, the `blue` band is a copy of green (see below). |
| `snaefellsjokull_<year>_candidates.png` | The clearest candidate days for that year, so you can pick a different one |
| `scenes.csv` | One row per downloaded scene: year asked for, year used, date, satellite, cloud over the glacier |

### Figure style

All figures use one minimal black-and-white style: white background, black text and lines, greys for secondary elements, no gridlines and no top or right axis borders. Satellite images keep their natural colours, because the colour is the data.

## 1. Setup

In [ ]:
# Install dependencies (run once)
%pip install -q earthengine-api pandas numpy matplotlib pillow requests rasterio

In [ ]:
from io import BytesIO
from pathlib import Path

import ee
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import requests
from IPython.display import display
from PIL import Image

In [ ]:
from cycler import cycler

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "text.color": "black",
    "axes.edgecolor": "black",
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "axes.prop_cycle": cycler(color=["black"]),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
    "axes.linewidth": 0.8,
    "axes.titlesize": 11,
    "axes.titlelocation": "left",
    "font.family": "sans-serif",
    "font.size": 10,
    "lines.linewidth": 1.2,
    "legend.frameon": False,
})

GREY = "#9a9a9a"
LIGHT_GREY = "#d4d4d4"

In [ ]:
# Authenticate and initialize Earth Engine.
# Use a Google Cloud project that has the Earth Engine API enabled.
PROJECT = "gsapp-map"

try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)

print("Earth Engine initialized")

## 2. Settings

- `YEARS`: the years you want. Add or remove years freely.
- `SEASON`: the window searched in each year. Late August to September is the end of the melt season, when the least seasonal snow hides the glacier, so it's the best time to see the ice itself.
- `SCENE_OVERRIDES`: after looking at the candidate sheets in section 5, pin a specific day for a year, e.g. `{2026: "2026-09-02"}`.

In [ ]:
YEARS = [1970, 2026]

# Search window within each year (month, day), inclusive
SEASON_START = (8, 1)
SEASON_END = (9, 30)

# If a year has no usable imagery in the season, use the nearest year that does (up to this many years away).
# "Usable" means at least one day with no more than MAX_CLOUD of the glacier under cloud.
USE_NEAREST_YEAR = True
MAX_YEAR_SHIFT = 10
MAX_CLOUD = 0.25

# Small preference for later days, so of two equally clear days the one with less seasonal snow wins
LATE_SEASON_WEIGHT = 0.05

# Pin a specific day for a year: {year: "YYYY-MM-DD"}. Leave empty to take the clearest day.
SCENE_OVERRIDES = {}

# Snæfellsjökull summit
SUMMIT = ee.Geometry.Point([-23.774, 64.808])

# Area to download: roughly 12 x 11 km around the ice cap (same box as v0_satellite_annual_composites.ipynb)
AOI = ee.Geometry.Rectangle([-23.90, 64.76, -23.65, 64.86])

# Area used to judge cloudiness: the ice cap and its moraines. Clouds over the sea don't matter.
GLACIER_REGION = SUMMIT.buffer(4500)

EXPORT_CRS = "EPSG:32627"  # UTM zone 27N
N_CANDIDATES = 6           # candidate days shown per year
CANDIDATE_PX = 360         # size of each candidate thumbnail

# Display: reflectance is linear, screens are not, so the PNG uses the standard sRGB curve.
# WHITE_REFLECTANCE is the reflectance shown as pure white. Fresh snow is ~0.9; lower it to brighten rock.
WHITE_REFLECTANCE = 1.0

GEOTIFF_NODATA = -32768

OUT_DIR = Path("outputs/true_color")
OUT_DIR.mkdir(parents=True, exist_ok=True)
SLUG = "snaefellsjokull"

## 3. Satellites

Every satellite is converted to the same four bands (`red, green, blue, nir`) in reflectance (0–1), plus a `cloud` band (1 = cloudy) used only for ranking days. Pixels are **never masked**, so the downloaded image is exactly what the satellite saw.

- **Sentinel-2** clouds come from Google's Cloud Score+ (`cs_cdf` below 0.6 = cloudy).
- **Landsat** clouds come from each scene's quality band (cloud, dilated cloud and cloud shadow flags). It sometimes calls bright snow "cloud", which only affects the ranking, not the image.
- **Landsat 7 after May 2003** has striped gaps (its Scan Line Corrector failed). Its scenes are only picked if nothing else is clear; `SLC_OFF_PENALTY` sets how strongly they are avoided.
- **Landsat MSS (1972–1983)** scenes are often poorly georeferenced; those with a geolocation error above `MSS_MAX_GEOMETRIC_RMSE` are skipped. MSS's cloud flag is also crude, so check its candidate sheet. MSS has green, red and two near-infrared bands, but no blue. The image uses red and green as they are and **copies green into blue**. Snow, ice and rock look close to natural; vegetation looks slightly cyan. MSS images are top-of-atmosphere reflectance, so they look hazier than the later surface-reflectance images.

In [ ]:
SLC_OFF_PENALTY = 0.25
SLC_FAILURE = "2003-05-31"

# Many MSS scenes are poorly georeferenced (Tier 2). Skip those whose geolocation error exceeds this (m).
MSS_MAX_GEOMETRIC_RMSE = 30


def landsat_cloud(qa, bits=(1, 3, 4)):
    cloud = ee.Image(0)
    for bit in bits:
        cloud = cloud.Or(qa.bitwiseAnd(1 << bit).neq(0))
    return cloud.rename("cloud")


def landsat_l2(coll_id, bands):
    """Landsat Collection 2 surface reflectance. `bands` = source names for red, green, blue, nir."""
    def prep(img):
        refl = img.select(bands, ["red", "green", "blue", "nir"]).multiply(0.0000275).add(-0.2)
        penalty = ee.Number(ee.Algorithms.If(
            ee.String(img.get("SPACECRAFT_ID")).equals("LANDSAT_7")
            .And(img.date().millis().gt(ee.Date(SLC_FAILURE).millis())),
            SLC_OFF_PENALTY, 0,
        ))
        return ee.Image(refl.addBands(landsat_cloud(img.select("QA_PIXEL")))
                        .copyProperties(img, ["system:time_start"])).set("penalty", penalty)
    return ee.ImageCollection(coll_id).map(prep)


def landsat_mss(coll_id, green, red, nir):
    """Landsat MSS, top-of-atmosphere reflectance. No blue band, so blue is a copy of green."""
    def prep(img):
        toa = ee.Algorithms.Landsat.TOA(img)
        refl = toa.select([red, green, green, nir], ["red", "green", "blue", "nir"])
        return ee.Image(refl.addBands(landsat_cloud(img.select("QA_PIXEL"), bits=(3,)))
                        .copyProperties(img, ["system:time_start"])).set("penalty", 0)
    return (ee.ImageCollection(coll_id)
            .filter(ee.Filter.lt("GEOMETRIC_RMSE_MODEL", MSS_MAX_GEOMETRIC_RMSE))
            .map(prep))


def sentinel2(coll_id):
    cs_plus = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")

    def prep(img):
        refl = img.select(["B4", "B3", "B2", "B8"], ["red", "green", "blue", "nir"]).divide(10000)
        cloud = img.select("cs_cdf").lt(0.6).rename("cloud")
        return ee.Image(refl.addBands(cloud).copyProperties(img, ["system:time_start"])).set("penalty", 0)
    return ee.ImageCollection(coll_id).linkCollection(cs_plus, ["cs_cdf"]).map(prep)


def merge(*collections):
    out = collections[0]
    for c in collections[1:]:
        out = out.merge(c)
    return out


# name, first year, last year, pixel size (m), true colour?, collection builder
SENSORS = [
    ("sentinel2", 2017, 9999, 10, True,
     lambda: sentinel2("COPERNICUS/S2_SR_HARMONIZED")),
    ("sentinel2-toa", 2015, 2016, 10, True,
     lambda: sentinel2("COPERNICUS/S2_HARMONIZED")),
    ("landsat", 1984, 2014, 30, True,
     lambda: merge(
         landsat_l2("LANDSAT/LT05/C02/T1_L2", ["SR_B3", "SR_B2", "SR_B1", "SR_B4"]),
         landsat_l2("LANDSAT/LE07/C02/T1_L2", ["SR_B3", "SR_B2", "SR_B1", "SR_B4"]),
         landsat_l2("LANDSAT/LC08/C02/T1_L2", ["SR_B4", "SR_B3", "SR_B2", "SR_B5"]),
     )),
    ("landsat-mss", 1972, 1983, 60, False,
     lambda: merge(
         *[landsat_mss(f"LANDSAT/LM0{n}/C02/{tier}", "B4", "B5", "B7") for n in (1, 2, 3) for tier in ("T1", "T2")],
     )),
]


def sensor_for(year):
    for name, first, last, pixel, true_colour, build in SENSORS:
        if first <= year <= last:
            return {"name": name, "pixel": pixel, "true_colour": true_colour, "collection": build()}
    return None

## 4. Rank the days in each year

Only scenes whose footprint contains the summit are used, and scenes from the same day are mosaicked together. For each day, two numbers are measured inside `GLACIER_REGION`:

- `cloud`: share of the region flagged as cloud
- `coverage`: share of the region with any data

Days are ranked by `cloud + (1 − coverage) + penalty`, lowest first, with a small bonus (`LATE_SEASON_WEIGHT`) for days later in the season.

If the requested year has no day with at most `MAX_CLOUD` cloud over the glacier, the years around it are searched in order of distance (±1, ±2, …). If none qualifies, the clearest day found is used.

In [ ]:
def season(year):
    start = ee.Date.fromYMD(year, *SEASON_START)
    end = ee.Date.fromYMD(year, *SEASON_END).advance(1, "day")
    return start, end


def day_mosaics(collection, year):
    """One mosaic per day in the season, with cloud, coverage and score properties."""
    start, end = season(year)
    # The scene's footprint must contain the summit; scenes that only clip the region's edge are skipped
    scenes = collection.filterBounds(SUMMIT).filterDate(start, end)
    days = scenes.aggregate_array("system:time_start").map(
        lambda t: ee.Date(t).format("YYYY-MM-dd")).distinct()

    def mosaic(day):
        d = ee.Date.parse("YYYY-MM-dd", day)
        same_day = scenes.filterDate(d, d.advance(1, "day"))
        img = same_day.mosaic()
        stats = img.select(["cloud"]).unmask(1).addBands(img.select("red").mask().rename("coverage")).reduceRegion(
            ee.Reducer.mean(), GLACIER_REGION, scale=60, maxPixels=1e8)
        cloud, coverage = ee.Number(stats.get("cloud")), ee.Number(stats.get("coverage"))
        penalty = ee.Number(same_day.aggregate_max("penalty"))
        earliness = end.difference(d, "day").divide(end.difference(start, "day"))  # 1 at start, 0 at end
        return img.set({
            "date": day,
            "system:time_start": d.millis(),
            "cloud": cloud,
            "coverage": coverage,
            "score": cloud.add(ee.Number(1).subtract(coverage)).add(penalty).add(earliness.multiply(LATE_SEASON_WEIGHT)),
        })

    return ee.ImageCollection(days.map(mosaic))


def rank_days(year):
    """Return (sensor, ranked list of {date, cloud, coverage, score}) for a year, or (None, [])."""
    sensor = sensor_for(year)
    if sensor is None:
        return None, []
    mosaics = day_mosaics(sensor["collection"], year)
    table = mosaics.sort("score").reduceColumns(
        ee.Reducer.toList(4), ["date", "cloud", "coverage", "score"]).get("list").getInfo()
    ranked = [dict(zip(["date", "cloud", "coverage", "score"], row)) for row in table]
    sensor["mosaics"] = mosaics
    return sensor, ranked


def find_year(year):
    """The year itself, or (if USE_NEAREST_YEAR) the nearest year with a day clear enough to use."""
    shifts = [0] + ([s for k in range(1, MAX_YEAR_SHIFT + 1) for s in (-k, k)] if USE_NEAREST_YEAR else [])
    clearest = (None, None, [])
    for shift in shifts:
        if year + shift > pd.Timestamp.now().year:
            continue
        sensor, ranked = rank_days(year + shift)
        if not ranked:
            continue
        if year in SCENE_OVERRIDES and shift == 0:
            return year, sensor, ranked
        if ranked[0]["cloud"] <= MAX_CLOUD:
            return year + shift, sensor, ranked
        if not clearest[2] or ranked[0]["cloud"] < clearest[2][0]["cloud"]:
            clearest = (year + shift, sensor, ranked)
    return clearest


choices = {}
for year in YEARS:
    used, sensor, ranked = find_year(year)
    if used is None:
        print(f"{year}: no imagery within {MAX_YEAR_SHIFT} years. Satellite imagery starts in 1972.")
        continue
    note = "" if used == year else f"  (no clear imagery for {year}; using {used})"
    if ranked[0]["cloud"] > MAX_CLOUD:
        note += f"  WARNING: clearest day is {ranked[0]['cloud']:.0%} cloudy"
    print(f"{year}: {len(ranked)} days with {sensor['name']} imagery in {used}{note}")
    for row in ranked[:N_CANDIDATES]:
        print(f"    {row['date']}  cloud {row['cloud']:.0%}  coverage {row['coverage']:.0%}")
    choices[year] = {"used": used, "sensor": sensor, "ranked": ranked}

## 5. Candidate sheets

The clearest `N_CANDIDATES` days for each year. The scene that will be downloaded is marked. To use a different day, add it to `SCENE_OVERRIDES` in section 2 and re-run from there.

In [ ]:
def to_display(img):
    """Reflectance (0–1, linear) to an 8-bit RGB image with the sRGB curve."""
    x = np.clip(img / WHITE_REFLECTANCE, 0, 1)
    srgb = np.where(x <= 0.0031308, 12.92 * x, 1.055 * np.power(x, 1 / 2.4) - 0.055)
    return np.round(srgb * 255).astype("uint8")


def thumbnail(mosaics, date, px):
    """Quick preview from Earth Engine. Gamma 2.2 approximates the sRGB curve used for the full images."""
    img = ee.Image(mosaics.filter(ee.Filter.eq("date", date)).first())
    url = img.visualize(bands=["red", "green", "blue"], min=0, max=WHITE_REFLECTANCE, gamma=2.2).getThumbURL({
        "region": AOI, "dimensions": px, "format": "png", "crs": EXPORT_CRS,
    })
    r = requests.get(url, timeout=300)
    r.raise_for_status()
    return Image.open(BytesIO(r.content)).convert("RGB")


def chosen_date(year, choice):
    return SCENE_OVERRIDES.get(year) or choice["ranked"][0]["date"]


for year, choice in choices.items():
    rows = choice["ranked"][:N_CANDIDATES]
    pick = chosen_date(year, choice)
    if pick not in [r["date"] for r in rows]:
        rows = rows + [r for r in choice["ranked"] if r["date"] == pick]

    fig, axes = plt.subplots(1, len(rows), figsize=(2.4 * len(rows), 2.9), squeeze=False)
    for ax, row in zip(axes[0], rows):
        ax.imshow(thumbnail(choice["sensor"]["mosaics"], row["date"], CANDIDATE_PX))
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(row["date"] == pick)
            spine.set_linewidth(2.5)
        ax.set_title(f"{row['date']}\ncloud {row['cloud']:.0%}" + ("  · chosen" if row["date"] == pick else ""),
                     fontsize=8, loc="left")
    used = choice["used"]
    fig.suptitle(f"{year}" + (f" → {used}" if used != year else "") + f" · {choice['sensor']['name']}",
                 x=0.01, ha="left", fontsize=11)
    fig.tight_layout()
    fig.savefig(OUT_DIR / f"{SLUG}_{year}_candidates.png", dpi=150)
    plt.show()

## 6. Download the chosen scenes

Each scene is downloaded as a GeoTIFF at the satellite's own pixel size, then turned into a PNG locally, one image pixel per satellite pixel. Nothing is masked, composited or sharpened.

In [ ]:
def fetch(url, path):
    r = requests.get(url, timeout=600)
    r.raise_for_status()
    path.write_bytes(r.content)


rows = []
for year, choice in choices.items():
    sensor, date = choice["sensor"], chosen_date(year, choice)
    info = next(r for r in choice["ranked"] if r["date"] == date)
    img = ee.Image(sensor["mosaics"].filter(ee.Filter.eq("date", date)).first())
    stem = f"{SLUG}_{year}_{sensor['name']}_{date}"

    scaled = (img.select(["red", "green", "blue", "nir"])
              .multiply(10000).round().toInt16().unmask(GEOTIFF_NODATA, False))
    url = scaled.getDownloadURL({
        "format": "GEO_TIFF", "region": AOI, "scale": sensor["pixel"], "crs": EXPORT_CRS,
    })
    tif_path = OUT_DIR / f"{stem}.tif"
    fetch(url, tif_path)

    with rasterio.open(tif_path, "r+") as ds:
        ds.nodata = GEOTIFF_NODATA
        for i, name in enumerate(["red", "green", "blue", "nir"], start=1):
            ds.set_band_description(i, name)
        ds.update_tags(date=date, sensor=sensor["name"], units="reflectance x 10000",
                       blue_band="copy of green (MSS has no blue band)" if not sensor["true_colour"] else "blue")
        rgb = ds.read([1, 2, 3]).astype("float64")
        nodata = (ds.read(1) == GEOTIFF_NODATA)

    rgb = np.moveaxis(np.where(rgb == GEOTIFF_NODATA, 0, rgb) / 10000, 0, -1)
    png = to_display(rgb)
    png[nodata] = 255
    Image.fromarray(png).save(OUT_DIR / f"{stem}.png")

    rows.append({
        "year_requested": year, "year_used": choice["used"], "date": date, "sensor": sensor["name"],
        "pixel_m": sensor["pixel"], "true_colour": sensor["true_colour"],
        "cloud_over_glacier": round(info["cloud"], 3), "coverage": round(info["coverage"], 3),
        "png": f"{stem}.png", "geotiff": f"{stem}.tif",
    })
    print(f"wrote {stem}.png ({png.shape[1]} x {png.shape[0]} px) and .tif")

scenes = pd.DataFrame(rows)
scenes.to_csv(OUT_DIR / "scenes.csv", index=False)
scenes

## 7. The scenes

In [ ]:
fig, axes = plt.subplots(1, len(rows), figsize=(6 * len(rows), 6.4), squeeze=False)
for ax, row in zip(axes[0], rows):
    ax.imshow(Image.open(OUT_DIR / row["png"]))
    ax.axis("off")
    label = str(row["year_used"]) if row["year_used"] == row["year_requested"] else \
        f"{row['year_used']} (nearest to {row['year_requested']})"
    ax.set_title(label, fontsize=14, fontweight="bold")
    caption = f"{row['sensor']} · {row['date']} · {row['pixel_m']} m pixels"
    if not row["true_colour"]:
        caption += " · approximate colour (no blue band)"
    ax.text(0, -0.02, caption, transform=ax.transAxes, va="top", fontsize=8, color=GREY)
fig.tight_layout()
plt.show()

## Older imagery

- **Aerial photographs** (Iceland's national archive, 1945–1991, 0.5–1 m, black and white) are much sharper than any satellite image of those years. `aerial_photos.ipynb` downloads them.
- **CORONA spy-satellite photographs** (USA, 1960–1972), declassified and free from [USGS EarthExplorer](https://earthexplorer.usgs.gov/) under *Declassified Data → Declass 1*. Black and white, about 2 m detail. They come as scanned film strips and must be georeferenced by hand, but they are the only satellite images from the 1960s.
- **Commercial imagery** (Maxar, Planet, Airbus) goes sharper than Sentinel-2's 10 m for recent years (0.3–3 m) but isn't free.

Colour satellite imagery of Snæfellsjökull starts in 1984 (Landsat 5).